# Day 22: RAG & Knowledge Grounding

**30-Day AI Agent Challenge: OpenAI Agents SDK vs LangGraph vs CrewAI**

---

## Today's Concept

> An agent that only knows what you tell it — that's knowledge grounding.

Retrieval-Augmented Generation (RAG) lets agents answer from YOUR data,
not just their training data. Today we add a knowledge base to agents in
all 3 frameworks — using simple keyword search (no vector DB required).

## What You'll Build
- A knowledge base of company policies (as a Python list)
- A retrieval function that finds relevant documents
- Agents grounded in your knowledge, not hallucinating

In [1]:
# ── Setup ───────────────────────────────────────────────
import sys
sys.path.insert(0, "..")
from shared import check_and_report, print_config, disable_openai_tracing, get_openai_agents_model, get_langgraph_model, get_crewai_llm
check_and_report()
disable_openai_tracing()
print("=" * 50)
print("DAY 22 SETUP COMPLETE")
print("=" * 50)
print_config()

Python: 3.12.13



All required packages installed.
Optional (missing): langchain-openai, langgraph-checkpoint-memory, ipywidgets

Ollama: connected (10 model(s) available)

DAY 22 SETUP COMPLETE
  Provider: OLLAMA
  Model:    qwen2.5:3b
  Ollama:   http://localhost:11434


In [2]:
# ── Knowledge Base ──────────────────────────────────────
# In production, this would be a vector DB (Chroma, Pinecone, etc.)
# For learning, a Python list with keyword search works perfectly.

import re

knowledge_base = [
    {
        "id": 1,
        "category": "returns",
        "content": "Return Policy: Items can be returned within 30 days of purchase. Items must be in original packaging with receipt. Refunds are processed within 5-7 business days."
    },
    {
        "id": 2,
        "category": "shipping",
        "content": "Shipping: Standard shipping takes 3-5 business days. Express shipping takes 1-2 business days. Free shipping on orders over $50."
    },
    {
        "id": 3,
        "category": "pricing",
        "content": "Pricing: We offer tiered pricing for bulk orders. 10+ items get 10% off, 50+ items get 20% off, 100+ items get 30% off. Contact sales@company.com for enterprise pricing."
    },
    {
        "id": 4,
        "category": "support",
        "content": "Support Hours: Live support available Monday-Friday 9 AM - 6 PM EST. Email support available 24/7 at help@company.com. Average response time: 2 hours."
    },
    {
        "id": 5,
        "category": "account",
        "content": "Account Management: Users can update their profile, change password, and manage billing in the Account Settings page. Two-factor authentication is available and recommended."
    },
    {
        "id": 6,
        "category": "returns",
        "content": "Exchanges: Exchanges are free for different sizes of the same product. Exchanges for different products follow the return policy and require a new purchase."
    },
]

def _tokens(text: str) -> set[str]:
    """Lowercase and split into a token set (drops punctuation)."""
    return set(re.findall(r"[a-z0-9]+", text.lower()))

def retrieve(query: str, top_k: int = 2) -> list[dict]:
    """Simple keyword-based retrieval (no vector DB needed for learning).

    Scores each document by how many query tokens it contains.
    Uses token-set intersection so short words like 'i' or 'a' can't
    match as substrings of longer words.
    """
    query_tokens = _tokens(query)
    scored = []
    for doc in knowledge_base:
        doc_tokens = _tokens(doc["content"]) | _tokens(doc["category"])
        overlap = query_tokens & doc_tokens
        if overlap:
            scored.append((doc, len(overlap)))
    scored.sort(key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in scored[:top_k]] if scored else knowledge_base[:top_k]

# Test retrieval
print("Retrieval test:")
for q in ["How do I return an item?", "shipping cost", "bulk discount", "free shipping"]:
    docs = retrieve(q)
    print(f"  Q: '{q}' → Found {len(docs)} docs (categories: {[d['category'] for d in docs]})")

Retrieval test:
  Q: 'How do I return an item?' → Found 2 docs (categories: ['returns', 'returns'])
  Q: 'shipping cost' → Found 1 docs (categories: ['shipping'])
  Q: 'bulk discount' → Found 1 docs (categories: ['pricing'])
  Q: 'free shipping' → Found 2 docs (categories: ['shipping', 'returns'])


---
## Framework 1: OpenAI Agents SDK — Tool-Based RAG

In [3]:
from agents import Agent, Runner, function_tool

model = get_openai_agents_model()

@function_tool
def search_knowledge(query: str) -> str:
    """Search the company knowledge base for relevant information."""
    docs = retrieve(query)
    return "\n".join(f"[{d['category'].upper()}] {d['content']}" for d in docs)

rag_agent = Agent(
    name="Knowledge Agent",
    instructions=(
        "You answer questions about company policies. "
        "ALWAYS use the search_knowledge tool first to find relevant information. "
        "Only answer based on the knowledge base results. "
        "If the knowledge base doesn't contain the answer, say so honestly."
    ),
    tools=[search_knowledge],
    model=model,
)

print("=" * 60)
print("OPENAI AGENTS SDK — RAG AGENT")
print("=" * 60)

questions = [
    "What's your return policy?",
    "Do you offer bulk discounts?",
    "Do you offer free shipping?",
]

for q in questions:
    print(f"\n>>> {q}")
    result = await Runner.run(rag_agent, q, max_turns=5)
    print(f"{result.final_output}\n")

OPENAI AGENTS SDK — RAG AGENT

>>> What's your return policy?


Our return policy allows you to return items within 30 days of purchase. Items must be in their original packaging with the receipt. Refunds will be processed within 5-7 business days.

For exchanges, if need to exchange for a different product or for products of equivalent value due to different sizes, this is free of charge. However, if you wish to exchange for another product or if the items exchanged are more expensive than your current purchase, a new item will need to be purchased first.


>>> Do you offer bulk discounts?


We offer tiered pricing for bulk orders. Orders of 10 or more items receive a 10% discount, orders of 50 or more items receive a 20% discount, and orders of 100 or more items receive a 30% discount. For enterprise-level discounts, you should contact sales@company.com.


>>> Do you offer free shipping?


We offer free shipping on orders over $50. Exchanges are currently free as they pertain to ordering the same size, and there are no additional fees for exchanging between similar products when you don't need a refund. Please note that exchanges for different products apply our regular return policy.

Remember also that standard delivery takes 3-5 business days while express delivery is available in 1-2 business days; free delivery is applied to orders meeting the minimum threshold of $50.



---
## Framework 2: LangGraph — RAG with State

In [4]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from typing_extensions import TypedDict

model = get_langgraph_model()

@tool
def search_docs(query: str) -> str:
    """Search company documents."""
    docs = retrieve(query)
    return "\n".join(f"[{d['category']}] {d['content']}" for d in docs)

class RAGState(TypedDict):
    query: str
    context: str
    answer: str

def retrieve_node(state: RAGState) -> dict:
    """Retrieve relevant documents."""
    docs = retrieve(state["query"])
    context = "\n".join(f"[{d['category']}] {d['content']}" for d in docs)
    print(f"  [RETRIEVE] Found {len(docs)} relevant documents")
    return {"context": context}

def answer_node(state: RAGState) -> dict:
    """Generate answer from retrieved context."""
    # Short, direct prompt — small local models anchor on refusal phrases,
    # so tell them the answer IS in the context rather than offering an out.
    r = model.invoke([
        SystemMessage(content=(
            "Using only the context below, answer the question. "
            "The answer is in the context."
        )),
        HumanMessage(content=(
            f"Context:\n{state['context']}\n\n"
            f"Question: {state['query']}"
        )),
    ])
    return {"answer": r.content}

builder = StateGraph(RAGState)
builder.add_node("retrieve", retrieve_node)
builder.add_node("answer", answer_node)
builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "answer")
builder.add_edge("answer", END)

graph = builder.compile()

print("=" * 60)
print("LANGGRAPH — RAG WITH EXPLICIT RETRIEVE → ANSWER")
print("=" * 60)

for q in questions:
    print(f"\n>>> {q}")
    result = graph.invoke({"query": q, "context": "", "answer": ""})
    print(f"{result['answer']}\n")

LANGGRAPH — RAG WITH EXPLICIT RETRIEVE → ANSWER

>>> What's your return policy?
  [RETRIEVE] Found 2 relevant documents


Items can be returned within 30 days of purchase. Items must be in original packaging with receipt. Refunds are processed within 5-7 business days.


>>> Do you offer bulk discounts?
  [RETRIEVE] Found 1 relevant documents


Yes, we do offer bulk discounts. The context mentions that there are tiered pricing options available for orders of 10 or more items, which gets a 10% discount; an order of 50 or more items gets a 20% discount and an order of 100 or more items gets a 30% discount.


>>> Do you offer free shipping?
  [RETRIEVE] Found 2 relevant documents


Free provisioning is available on orders over $50.



---
## Framework 3: CrewAI — RAG Crew

In [5]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

llm = get_crewai_llm()

@tool("Search Knowledge Base")
def search_kb_tool(query: str) -> str:
    """Search the company knowledge base for policy information."""
    docs = retrieve(query)
    return "\n".join(f"[{d['category']}] {d['content']}" for d in docs)

researcher = Agent(
    role="Knowledge Retriever",
    goal="Find relevant documents by calling search_kb_tool with the customer's exact question",
    backstory=(
        "You call the search_kb_tool once, passing the customer's original "
        "question unchanged. Keyword search works best with the customer's "
        "own words, so do not paraphrase or drop words like 'shipping' or 'free'."
    ),
    tools=[search_kb_tool],
    llm=llm, verbose=False,
)

responder = Agent(
    role="Customer Responder",
    goal="Answer the customer's question from the retrieved context",
    backstory=(
        "The answer to the customer's question is in the retrieved context. "
        "Find it there and state it. Do not say the information is missing "
        "if the context addresses the question in any way."
    ),
    llm=llm, verbose=False,
)

print("=" * 60)
print("CREWAI — RAG CREW")
print("=" * 60)

for q in questions:
    print(f"\n>>> {q}")

    retrieve_task = Task(
        description=f"Search the knowledge base for: {q}",
        expected_output="Relevant policy documents",
        agent=researcher,
    )

    answer_task = Task(
        description=f"Answer this customer question using the retrieved context: {q}",
        expected_output="Accurate answer based on knowledge base",
        agent=responder,
        context=[retrieve_task],
    )

    crew = Crew(
        agents=[researcher, responder],
        tasks=[retrieve_task, answer_task],
        process=Process.sequential,
        verbose=False,
    )
    result = await crew.kickoff_async()
    print(f"{str(result)}\n")

CREWAI — RAG CREW

>>> What's your return policy?


Items can be returned within 30 days of purchase, provided they are in their original packaging with the receipt. Refunds will be processed within 5-7 business days. Exchanges are offered free of charge for different sizes of the same product and follow the return policy when it comes to items being exchanged for different products. A new purchase is required for such exchanges.


>>> Do you offer bulk discounts?


Accurate answer based on knowledge base:
The company offers tiered pricing for bulk orders. Orders of 10 or more items come with a 10% discount, orders of 50 or more items receive a 20% discount, and orders of 100 or more items get a 30% discount. For enterprise-level bulk discounts, customers should contact sales@company.com.


>>> Do you offer free shipping?


This context does not contain a direct answer to the query "Do you offer free" services or any related information. Please check another part of the document or provide additional context.



---
## Comparison: RAG Approaches

| Framework | RAG Pattern | Retrieval Control | Grounding |
|---|---|---|---|
| OpenAI SDK | Tool-based (agent decides when) | Agent-driven | Via instructions |
| LangGraph | Explicit retrieve→answer graph | Deterministic | Via context injection |
| CrewAI | Researcher + Responder crew | Task-driven | Via context=[task] |

**Key insight:** LangGraph gives the most control over the RAG pipeline —
the retrieve step is ALWAYS called before answering. OpenAI SDK trusts
the agent to call the tool. CrewAI splits it into two roles.

**Production upgrade path:** Replace `retrieve()` with:
- **Vector DB** (ChromaDB, Pinecone) for semantic search
- **Embeddings** (OpenAI, sentence-transformers) for better matching
- **Re-ranking** for more relevant results
- **Hybrid search** (keyword + semantic) for best coverage

## Key Takeaway

RAG is just "retrieve then answer" — the pattern is the same across frameworks.
What differs is WHO controls the retrieval:
- **LangGraph**: The graph always retrieves first (deterministic)
- **OpenAI SDK**: The agent decides when to search (flexible but less reliable)
- **CrewAI**: A dedicated retriever agent always searches (role-based)

LangGraph's explicit pipeline gives the strongest *retrieval* guarantee — the
retrieve node runs before the answer node every time. That is not the same as
an *answer-quality* guarantee: the model still has to read the context and use
it, so pair the pipeline with a model capable of doing that. On a small local
model, the extra agent hop in the CrewAI pattern is where that signal is most
easily lost — in this notebook's run, CrewAI grounded 2 of 3 questions; the
shipping question was retrieved correctly but the responder dropped it.

---

